# 🧪 02 · AI Test-Case Generator

**Level:** Everyone · **Time:** ~20 min

Give the AI a **requirement / user story** → get back structured **test cases**:
positive, negative, edge, and boundary — as a clean table you can export.

Free, in the cloud, no API key.  ▶️ Run each cell with **Shift+Enter**.

> ⚙️ Enable GPU: *Runtime → Change runtime type → T4 GPU*.

### Step 1 — Setup (install + load model + logging)

In [ ]:
!pip install -q -U transformers accelerate
import torch, json, re, logging
from transformers import pipeline

# --- logging: every run is logged here AND to a file you can download ---
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s | %(levelname)s | %(message)s',
                    handlers=[logging.StreamHandler(),
                              logging.FileHandler('qa_tool.log')])
log = logging.getLogger('qa-gen')

MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'  # tip: try 'Qwen/Qwen2.5-3B-Instruct' for better quality
chat = pipeline('text-generation', model=MODEL,
                torch_dtype=torch.bfloat16, device_map='auto')
log.info('Model loaded: %s', MODEL)
print('✅ ready — logs are saved to qa_tool.log (Files panel on the left)')

### Step 2 — The QA prompt
This is the heart of the tool. We give the model a **clear role**, **exact instructions**,
and ask it to reply in **JSON** so we can turn it into a table. This is the part your
community should study and tweak — better prompt = better test cases.

In [ ]:
SYSTEM = '''You are a senior QA engineer. You write thorough, practical test cases.
For a given requirement, produce test cases covering POSITIVE, NEGATIVE, EDGE, and BOUNDARY scenarios.
Return ONLY valid JSON (no prose, no markdown fences) shaped exactly as:
{"test_cases": [
  {"id": "TC-01", "title": "...", "type": "positive|negative|edge|boundary",
   "priority": "high|medium|low", "steps": ["..."], "expected": "..."}
]}'''

def generate_test_cases(requirement, max_new_tokens=1200):
    log.info('Generating for requirement (%d chars)...', len(requirement))
    try:
        user = f'Requirement:\n{requirement}\n\nGenerate 6-10 test cases as JSON.'
        out = chat([{'role':'system','content':SYSTEM}, {'role':'user','content':user}],
                   max_new_tokens=max_new_tokens, do_sample=False)
        text = out[0]['generated_text'][-1]['content']
    except Exception as e:
        log.error('Model call failed: %s', e)
        return {'test_cases': [], 'error': str(e)}
    # be forgiving: pull the JSON object out even if the model adds stray text
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if not m:
        log.warning('No JSON found in output — keeping raw text for inspection.')
        return {'test_cases': [], 'raw': text}
    try:
        data = json.loads(m.group(0))
        log.info('Parsed %d test cases', len(data.get('test_cases', [])))
        return data
    except json.JSONDecodeError as e:
        log.error('JSON parse failed: %s — re-run the cell or try the 3B model.', e)
        return {'test_cases': [], 'raw': text, 'error': str(e)}

print('✅ generator ready')

### ⭐ Easy mode — type your requirement in the box (no code!)
Run this cell once. A **text box** and a **button** appear. Type or paste your
requirement (multiple lines are fine — no quotes to worry about), then click
**Generate**. This is the friendliest way for everyone in the room.

In [ ]:
# 👇 You don't need to read or edit this code — just press Shift+Enter to run it.
#    A text box + button will appear below. Type your requirement there and click Generate.
import ipywidgets as widgets
import pandas as pd
from IPython.display import display, clear_output

box = widgets.Textarea(
    value=('Given the customer is browsing available products\n'
           'When they add "Wireless Headphones" to their cart\n'
           'Then the cart should contain 1 item\n'
           'And the cart total should be 149.99'),
    placeholder='Paste a requirement or user story here...',
    layout=widgets.Layout(width='100%', height='130px'))
go = widgets.Button(description='Generate test cases ✨', button_style='primary')
out = widgets.Output()

def on_click(_):
    with out:
        clear_output()
        if not box.value.strip():
            print('Please type a requirement first.'); return
        print('⏳ Generating...')
        result = generate_test_cases(box.value)
        clear_output()
        if result.get('error'):
            print('⚠️ Error:', result['error']); return
        if not result['test_cases']:
            print('⚠️ No test cases parsed. Raw output:'); print(result.get('raw','')); return
        df = pd.DataFrame(result['test_cases'])
        if 'steps' in df:
            df['steps'] = df['steps'].apply(lambda s: ' → '.join(s) if isinstance(s, list) else s)
        print(f'✅ {len(df)} test cases:')
        display(df)

go.on_click(on_click)
display(box, go, out)

---
### 🧑‍💻 Code mode (optional) — for those who want to see how it works
The cells below do the same thing in plain code. Great for learning, but if you
edited the text, remember: **multi-line text needs triple quotes** `"""..."""`.

### Step 3 — Generate! ✨
Edit `requirement` to anything from your own backlog.

> ⚠️ **Important:** keep the **triple quotes** `""" ... """`. They let you write a
> requirement across **multiple lines** (e.g. Gherkin Given/When/Then). If you use
> single quotes `'...'` and press Enter inside, Python throws
> `SyntaxError: unterminated string literal`. Triple quotes fix that.

In [ ]:
# Multi-line is fine inside triple quotes:
requirement = """Given the customer is browsing available products
When they add an item to the cart
Then the cart total should update correctly"""

result = generate_test_cases(requirement)
if result.get('error'):
    print('⚠️ Something went wrong — check the log above. Error:', result['error'])
elif not result['test_cases']:
    print('⚠️ No test cases parsed. Raw model output:'); print(result.get('raw',''))
else:
    print(f"✅ Generated {len(result['test_cases'])} test cases")

### Step 4 — See them as a table

In [ ]:
import pandas as pd
df = pd.DataFrame(result['test_cases'])
if 'steps' in df: df['steps'] = df['steps'].apply(lambda s: ' → '.join(s) if isinstance(s, list) else s)
df

### Step 5 — Export (Markdown / CSV)
Take the output straight into Jira, TestRail, a wiki, or a spreadsheet.

In [ ]:
df.to_csv('test_cases.csv', index=False)
with open('test_cases.md','w') as f: f.write(df.to_markdown(index=False))
print('✅ saved test_cases.csv and test_cases.md  (see the Files panel on the left)')
print(df.to_markdown(index=False))

---
### 🧠 Discussion for the group
- Did it find edge cases *you* would have missed? Did it invent any wrong ones?
- **This is the key lesson:** AI is a *fast first draft*, not a replacement for a tester's judgement.
- Try improving `SYSTEM` (e.g. ask for Gherkin, or for test data values). Re-run and compare.

👉 Next: **`03_custom_qa_assistant.ipynb`** — shape the model into a dedicated QA-Bot.